In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from tensorflow.keras.layers import Dense, Input, LSTM, SimpleRNN
from tensorflow.keras.models import Sequential

# Find the repository root whether Jupyter was started from the root or Model/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "Dataset").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_PATH = PROJECT_ROOT / "Dataset" / "CarPrice_Assignment.csv"
IMAGES_DIR = PROJECT_ROOT / "Images"
MODEL_DIR = PROJECT_ROOT / "Model"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

data = pd.read_csv(DATASET_PATH)
df = data.dropna(subset=["price"]).drop(columns=["car_ID", "CarName"]).copy()


In [ ]:
df.head(10)

In [ ]:
# Pair plot for numerical features
pairplot_columns = [
    "wheelbase", "carlength", "carwidth", "carheight",
    "curbweight", "enginesize", "horsepower", "price",
]
sns.pairplot(data[pairplot_columns])
plt.savefig(IMAGES_DIR / "PairPlot.png", bbox_inches="tight")
plt.show()


In [ ]:
# Correlation heatmap for numeric features
plt.figure(figsize=(16, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Heatmap")
plt.savefig(IMAGES_DIR / "CorrelationHeatmap.png", bbox_inches="tight")
plt.show()


In [ ]:
# Box plot for car body vs. price
plt.figure(figsize=(12, 8))
sns.boxplot(x="carbody", y="price", data=df)
plt.title("Box Plot: Car Body vs. Price")
plt.savefig(IMAGES_DIR / "box_CarBody_vs_Price.png", bbox_inches="tight")
plt.show()


In [ ]:
# Interactive 3D scatter plot for curb weight, horsepower, and price
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=data["curbweight"],
            y=data["horsepower"],
            z=data["price"],
            mode="markers",
            marker={"size": 8, "color": "blue", "opacity": 0.5},
            text=data["CarName"],
            hoverinfo="text",
        )
    ]
)
fig.update_layout(
    scene={
        "xaxis_title": "Curb Weight",
        "yaxis_title": "Horsepower",
        "zaxis_title": "Price",
    },
    title="Interactive 3D Scatter Plot: Curb Weight, Horsepower, and Price",
    margin={"l": 0, "r": 0, "b": 0, "t": 40},
)
fig.write_html(IMAGES_DIR / "interactive_3d_scatter_plot.html")
fig.show()


In [ ]:
# Split the data before fitting preprocessing to avoid leakage.
X = df.drop(columns="price")
y = df["price"].astype("float32")
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
    ],
    verbose_feature_names_out=False,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train_scaled = preprocessor.fit_transform(X_train).astype(np.float32)
X_test_scaled = preprocessor.transform(X_test).astype(np.float32)


In [ ]:
# Store error values for each model
error_values = []

 ## Simple Neural Network

In [ ]:
# Build and train a simple feed-forward neural network.
model_simple = Sequential(
    [
        Input(shape=(X_train_scaled.shape[1],)),
        Dense(64, activation="relu"),
        Dense(32, activation="relu"),
        Dense(1),
    ]
)
model_simple.compile(optimizer="adam", loss="mean_squared_error", metrics=["mean_squared_error"])
history_simple = model_simple.fit(
    X_train_scaled, y_train, epochs=100, batch_size=16, validation_split=0.2, verbose=0
)
y_pred_simple = model_simple.predict(X_test_scaled, verbose=0).ravel()
mse_simple = mean_squared_error(y_test, y_pred_simple)
r2_simple = r2_score(y_test, y_pred_simple)
error_values.append(("Simple Neural Network", mse_simple, r2_simple))
print(f"Simple Neural Network MSE: {mse_simple}, R-squared: {r2_simple}")


## Complex Neural Network

In [ ]:
# Build and train the deeper neural network.
model_complex = Sequential(
    [
        Input(shape=(X_train_scaled.shape[1],)),
        Dense(128, activation="relu"),
        Dense(64, activation="relu"),
        Dense(32, activation="relu"),
        Dense(1),
    ]
)
model_complex.compile(optimizer="adam", loss="mean_squared_error", metrics=["mean_squared_error"])
history_complex = model_complex.fit(
    X_train_scaled, y_train, epochs=100, batch_size=16, validation_split=0.2, verbose=0
)
y_pred_complex = model_complex.predict(X_test_scaled, verbose=0).ravel()
mse_complex = mean_squared_error(y_test, y_pred_complex)
r2_complex = r2_score(y_test, y_pred_complex)
error_values.append(("Complex Neural Network", mse_complex, r2_complex))
print(f"Complex Neural Network MSE: {mse_complex}, R-squared: {r2_complex}")


## MLP

In [ ]:
# The MLP uses the same compact feed-forward architecture as the baseline.
model_mlp = Sequential(
    [
        Input(shape=(X_train_scaled.shape[1],)),
        Dense(64, activation="relu"),
        Dense(32, activation="relu"),
        Dense(1),
    ]
)
model_mlp.compile(optimizer="adam", loss="mean_squared_error", metrics=["mean_squared_error"])
history_mlp = model_mlp.fit(
    X_train_scaled, y_train, epochs=100, batch_size=16, validation_split=0.2, verbose=0
)
y_pred_mlp = model_mlp.predict(X_test_scaled, verbose=0).ravel()
mse_mlp = mean_squared_error(y_test, y_pred_mlp)
r2_mlp = r2_score(y_test, y_pred_mlp)
error_values.append(("MLP", mse_mlp, r2_mlp))
print(f"MLP MSE: {mse_mlp}, R-squared: {r2_mlp}")


## RNN

In [ ]:
# Treat the ordered feature vector as a sequence for the SimpleRNN.
X_train_rnn = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_rnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)
model_rnn = Sequential(
    [
        Input(shape=(X_train_rnn.shape[1], 1)),
        SimpleRNN(64),
        Dense(32, activation="relu"),
        Dense(1),
    ]
)
model_rnn.compile(optimizer="adam", loss="mean_squared_error", metrics=["mean_squared_error"])
history_rnn = model_rnn.fit(
    X_train_rnn, y_train, epochs=100, batch_size=16, validation_split=0.2, verbose=0
)
y_pred_rnn = model_rnn.predict(X_test_rnn, verbose=0).ravel()
mse_rnn = mean_squared_error(y_test, y_pred_rnn)
r2_rnn = r2_score(y_test, y_pred_rnn)
error_values.append(("Simple RNN", mse_rnn, r2_rnn))
print(f"Simple RNN MSE: {mse_rnn}, R-squared: {r2_rnn}")


## LSTM

In [ ]:
# Reshape the tabular features into a sequence for the LSTM.
X_train_lstm = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_lstm = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)
model_lstm = Sequential(
    [
        Input(shape=(X_train_lstm.shape[1], 1)),
        LSTM(64),
        Dense(1),
    ]
)
model_lstm.compile(optimizer="adam", loss="mean_squared_error", metrics=["mean_squared_error"])
history_lstm = model_lstm.fit(
    X_train_lstm, y_train, epochs=100, batch_size=16, validation_split=0.2, verbose=0
)
y_pred_lstm = model_lstm.predict(X_test_lstm, verbose=0).ravel()
mse_lstm = mean_squared_error(y_test, y_pred_lstm)
r2_lstm = r2_score(y_test, y_pred_lstm)
error_values.append(("LSTM", mse_lstm, r2_lstm))
print(f"LSTM MSE: {mse_lstm}, R-squared: {r2_lstm}")


In [ ]:
# Save the models in Keras' native format.
models_to_save = [
    ("simple_neural_network", model_simple),
    ("complex_neural_network", model_complex),
    ("lstm", model_lstm),
    ("simple_rnn", model_rnn),
    ("mlp", model_mlp),
]
for model_name, model in models_to_save:
    model.save(MODEL_DIR / f"{model_name}.keras")
    print(f"{model_name} model saved successfully.")


In [ ]:
# Compare the mean squared error and R-squared scores.
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5))
model_names = [name for name, _, _ in error_values]
axes[0].bar(model_names, [mse for _, mse, _ in error_values], color="blue", width=0.6)
axes[0].set_title("Mean Squared Error (MSE) for Each Model")
axes[0].set_ylabel("MSE")
axes[0].tick_params(axis="x", labelrotation=45)
axes[1].bar(model_names, [r2 for _, _, r2 in error_values], color="green", width=0.6)
axes[1].set_title("R-squared for Each Model")
axes[1].set_ylabel("R-squared")
axes[1].tick_params(axis="x", labelrotation=45)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "ErrorValuesBarPlot.png", bbox_inches="tight")
plt.show()


In [ ]:
# Visualize training and validation loss for each model.
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 12))
models_and_histories = [
    ("Simple Neural Network", history_simple),
    ("Complex Neural Network", history_complex),
    ("LSTM", history_lstm),
    ("Simple RNN", history_rnn),
    ("MLP", history_mlp),
]
for index, (model_name, history) in enumerate(models_and_histories):
    axis = axes[index // 3, index % 3]
    axis.plot(history.history["loss"], label="Training Loss")
    axis.plot(history.history["val_loss"], label="Validation Loss")
    axis.set_title(model_name)
    axis.set_xlabel("Epochs")
    axis.set_ylabel("Loss")
    axis.legend()
fig.delaxes(axes[1, 2])
plt.tight_layout()
plt.savefig(IMAGES_DIR / "TrainingHistoryPlot.png", bbox_inches="tight")
plt.show()


In [ ]:
# Plot predictions from all models against the held-out actual prices.
models = [
    ("Simple Neural Network", model_simple, X_test_scaled),
    ("Complex Neural Network", model_complex, X_test_scaled),
    ("LSTM", model_lstm, X_test_lstm),
    ("Simple RNN", model_rnn, X_test_rnn),
    ("MLP", model_mlp, X_test_scaled),
]
plt.figure(figsize=(16, 8))
for model_name, model, X_eval in models:
    y_pred = model.predict(X_eval, verbose=0).ravel()
    plt.scatter(y_test, y_pred, label=model_name)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Predicted vs Actual Prices for All Models")
plt.legend()
plt.savefig(IMAGES_DIR / "Scatter_Predicted_vs_Actual.png", bbox_inches="tight")
plt.show()


In [ ]:
# Select the model with the lowest held-out mean squared error.
best_model_name, best_mse, best_r2 = min(error_values, key=lambda result: result[1])
models_by_name = {
    "Simple Neural Network": model_simple,
    "Complex Neural Network": model_complex,
    "MLP": model_mlp,
    "Simple RNN": model_rnn,
    "LSTM": model_lstm,
}
best_model = models_by_name[best_model_name]
print(f"Best model by MSE: {best_model_name} (MSE: {best_mse}, R-squared: {best_r2})")


In [ ]:
best_model.save(MODEL_DIR / "best_model.keras")
print(f"Best model ({best_model_name}) saved successfully!")
